# Module 8: Window Functions for Analytics

**ALY 6420 | OVER, PARTITION BY, ranking, frames, running calculations, LAG/LEAD, and top-N analysis**

*Course Lecture Notes*


## Module 8

### Keep the row — add the context

Module 7 introduced aggregation: many rows go in, one summary row per group comes out.

Window functions solve a different problem. They let you calculate a value **across related rows while preserving the individual rows**.

That makes questions such as these possible in one result set:

- What is this customer's payment, and what is the customer's lifetime total?
- Where does this customer rank inside their store?
- How much revenue had accumulated by this month?
- How does this month's revenue compare with the prior month?
- What is the moving average around this observation?

The defining idea is simple:

> **GROUP BY changes the number of rows. Window functions usually add analytical columns without collapsing the rows.**


## Learning objectives

By the end of this lecture, you should be able to:

- explain how window functions differ from `GROUP BY`
- identify when row preservation is necessary
- write window functions using `OVER()`
- use `PARTITION BY` to create independent calculation groups
- explain the difference between window `ORDER BY` and final-query `ORDER BY`
- apply `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, and `NTILE()`
- explain how each ranking function treats ties
- implement top-N-per-group patterns
- use aggregate functions such as `SUM`, `AVG`, and `COUNT` as window functions
- write running totals and moving averages using window frames
- use `LAG()` and `LEAD()` for adjacent-row and period-over-period comparisons
- use a named `WINDOW` definition to reduce repetition
- recognize common window-function errors involving grain, ties, filtering, and ordering


## Principal source and practice environment

### Principal resource

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics* (4th ed.), **Chapter 9: Inter-Row Operation with Window Functions**.

The textbook develops window analysis by moving from:

1. a row's own value,
2. to its position relative to other rows,
3. to partitions,
4. to ordering within a partition,
5. to reusable window definitions, and
6. to frames for rolling calculations.

### Practice environment

Examples in this lecture use the **Pagila** PostgreSQL database in DBeaver, primarily:

- `payment`
- `customer`
- `rental`
- `inventory`
- `film`
- `category`
- `film_category`


## A mental model for this module

For every window expression, identify four things:

```text
FUNCTION
   OVER (
       PARTITION BY ...
       ORDER BY ...
       ROWS/RANGE ...
   )
```

Ask:

1. **What calculation is being performed?**
2. **Which rows belong in the same partition?**
3. **In what order should those rows be evaluated?**
4. **How much of that ordered partition belongs in the current frame?**

Not every window needs all four pieces, but every correct window has a deliberate answer to those questions.


# Part 1: Why Window Functions Exist


## Three families of calculations

A useful progression from the textbook is:

### Scalar calculation
Uses values from the current row.

```sql
SELECT amount,
       amount * 1.10 AS amount_with_markup
FROM payment;
```

### Aggregate calculation
Uses many rows and collapses them.

```sql
SELECT customer_id,
       SUM(amount) AS total_paid
FROM payment
GROUP BY customer_id;
```

### Window calculation
Uses many related rows but keeps the current row.

```sql
SELECT customer_id,
       payment_date,
       amount,
       SUM(amount) OVER (
           PARTITION BY customer_id
       ) AS total_paid
FROM payment;
```


## Same data, different result grain

Compare these two queries.

### GROUP BY

```sql
SELECT customer_id,
       SUM(amount) AS total_paid
FROM payment
GROUP BY customer_id;
```

Result grain:

> one row per customer

### Window function

```sql
SELECT customer_id,
       payment_id,
       amount,
       SUM(amount) OVER (
           PARTITION BY customer_id
       ) AS total_paid
FROM payment;
```

Result grain:

> one row per payment

The customer's total is repeated beside each payment row.


## Quick check 1

**Think first.**

If the business question requires the original payment amount and the customer's total spending on the same row, should you begin with `GROUP BY` or a window function?

<details>
<summary>Answer</summary>

A window function. `GROUP BY` would collapse the payment rows, while a window function can preserve each payment and add the customer's total as another column.

</details>


## Window calculations do not create new source rows

A window function evaluates a set of existing result rows.

It does **not**:

- join in new rows,
- change foreign-key relationships,
- eliminate fan-out from a bad join,
- automatically change the query's base grain.

This matters because a window function can faithfully calculate over the wrong row set if the preceding joins are wrong.


## A useful rule

Before adding a window function, first state:

> **What does one row represent before the window is computed?**

If the answer is wrong, the window result will be wrong too.


# Part 2: The OVER() Clause


## `OVER()` turns a compatible function into a window calculation

The textbook presents a general pattern:

```sql
SELECT ...,
       window_function(...) OVER (
           PARTITION BY ...
           ORDER BY ...
       )
FROM ...;
```

The keyword `OVER` tells PostgreSQL:

> Calculate this function across a related set of rows rather than reducing the result to one grouped row.


## Empty `OVER()`: use the whole result set

```sql
SELECT
    payment_id,
    customer_id,
    amount,
    SUM(amount) OVER () AS grand_total
FROM payment;
```

Every payment row remains.

The same grand total is displayed on every row because the window contains the entire qualifying result set.


## Build a percentage of total

Once the grand total is available beside each row, you can compare an individual value to the whole.

```sql
SELECT
    payment_id,
    amount,
    SUM(amount) OVER () AS grand_total,
    ROUND(
        100.0 * amount / SUM(amount) OVER (),
        4
    ) AS pct_of_all_revenue
FROM payment;
```

This pattern is common in contribution and share-of-total analysis.


## `PARTITION BY`: create independent windows

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
    ) AS customer_total
FROM payment;
```

Each customer is a separate partition.

A payment for customer 1 cannot contribute to customer 2's total.


## PARTITION BY does not collapse

This is the key difference from `GROUP BY`.

```sql
GROUP BY customer_id
```

creates one output row per customer.

```sql
OVER (PARTITION BY customer_id)
```

keeps the payment rows and performs the calculation independently inside each customer's rows.


## Quick check 2

**Think first.**


What is the result grain here?

```sql
SELECT payment_id,
       customer_id,
       SUM(amount) OVER (PARTITION BY customer_id)
FROM payment;
```


<details>
<summary>Answer</summary>

One row per payment, because the window function does not collapse the payment rows.

</details>


## Multiple partition columns

Partitions can be defined by combinations of columns.

```sql
SELECT
    customer_id,
    staff_id,
    payment_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id, staff_id
    ) AS customer_staff_total
FROM payment;
```

Now each unique `(customer_id, staff_id)` combination has an independent window.


# Part 3: ORDER BY Inside OVER()


## Window ordering is calculation ordering

This query:

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date
    ) AS running_total
FROM payment;
```

does more than sort rows.

The `ORDER BY` inside `OVER()` defines the sequence in which the window calculation progresses.


## Final ORDER BY is different

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date
    ) AS running_total
FROM payment
ORDER BY customer_id, payment_date;
```

There are two orderings here:

- `ORDER BY payment_date` **inside `OVER()`** → controls the running calculation
- final `ORDER BY customer_id, payment_date` → controls presentation of the output

Do not treat them as interchangeable.


## Deterministic ordering

Suppose two payments from the same customer have the same timestamp.

If your calculation needs an exact row-by-row order, add a stable tiebreaker:

```sql
SUM(amount) OVER (
    PARTITION BY customer_id
    ORDER BY payment_date, payment_id
)
```

A deterministic order is especially important for `ROW_NUMBER()`, running totals, and `LAG()`.


## Quick check 3

**Think first.**

Why might `ORDER BY payment_date` be insufficient for a deterministic row-by-row calculation?

<details>
<summary>Answer</summary>

Because multiple rows can share the same payment date/time. Adding a unique tiebreaker such as `payment_id` creates a fully defined order.

</details>


# Part 4: Aggregate Functions as Window Functions


## Aggregates can become windows

The textbook emphasizes that common aggregate functions can also operate as window functions.

```sql
SELECT
    customer_id,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
    ) AS customer_total,
    AVG(amount) OVER (
        PARTITION BY customer_id
    ) AS customer_avg,
    COUNT(*) OVER (
        PARTITION BY customer_id
    ) AS customer_payment_count
FROM payment;
```

Each payment row now carries group context.


## Compare a row with its group average

```sql
SELECT
    customer_id,
    payment_id,
    amount,
    AVG(amount) OVER (
        PARTITION BY customer_id
    ) AS customer_avg,
    amount - AVG(amount) OVER (
        PARTITION BY customer_id
    ) AS difference_from_customer_avg
FROM payment;
```

This asks:

> Is this payment larger or smaller than the customer's typical payment?


## Compare a row with the global average

```sql
SELECT
    payment_id,
    amount,
    AVG(amount) OVER () AS overall_avg,
    amount - AVG(amount) OVER () AS difference_from_overall_avg
FROM payment;
```

Empty `OVER()` means the calculation uses one partition containing all qualifying rows.


## Same function, different analytical level

```sql
AVG(amount) OVER ()
```

means:

> average across all result rows

```sql
AVG(amount) OVER (PARTITION BY customer_id)
```

means:

> average within each customer

```sql
AVG(amount) OVER (
    PARTITION BY customer_id
    ORDER BY payment_date
)
```

means:

> cumulative average as the customer's ordered payment history unfolds

The function is the same. The window definition changes the question.


# Part 5: Ranking Functions


## Ranking asks about position

Ranking functions answer questions such as:

- Which customers spend the most?
- What are the top three customers in each store?
- Which films rank highest by rental count?
- How should customers be divided into quartiles?

The ranking is determined by the `ORDER BY` inside the window.


## `ROW_NUMBER()`

`ROW_NUMBER()` assigns a unique sequence:

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS payment_sequence
FROM payment;
```

Each customer's first payment gets 1, second gets 2, and so on.


## `ROW_NUMBER()` and ties

`ROW_NUMBER()` never gives two rows the same number.

If the `ORDER BY` columns tie and you do not supply a tiebreaker, PostgreSQL must still choose an order.

For reproducible results:

```sql
ORDER BY score DESC, customer_id
```

rather than only:

```sql
ORDER BY score DESC
```


## `RANK()`

`RANK()` gives tied rows the same rank and leaves gaps after the tie.

For values:

```text
100, 90, 90, 80
```

the ranks are:

```text
1, 2, 2, 4
```


## `DENSE_RANK()`

`DENSE_RANK()` also gives tied rows the same rank, but does not leave gaps.

For:

```text
100, 90, 90, 80
```

the dense ranks are:

```text
1, 2, 2, 3
```


## Ranking customer totals

First aggregate to one row per customer:

```sql
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(amount) AS total_paid
    FROM payment
    GROUP BY customer_id
)
SELECT
    customer_id,
    total_paid,
    RANK() OVER (
        ORDER BY total_paid DESC
    ) AS spending_rank
FROM customer_totals
ORDER BY spending_rank, customer_id;
```

The CTE fixes the grain before ranking.


## Why aggregate before ranking?

If you rank raw payment rows by `amount`, you rank individual payments.

If the question is:

> Which customers have spent the most overall?

you need one total per customer first.

Correct sequence:

```text
raw payments
   ↓ GROUP BY customer
customer totals
   ↓ window ranking
customer spending ranks
```


## Quick check 4

**Think first.**

You want to rank customers by lifetime spending. Should `RANK()` be applied directly to raw `payment` rows?

<details>
<summary>Answer</summary>

No. First aggregate payments to one row per customer, then rank those customer totals.

</details>


## Choosing among ranking functions

| Need | Function |
|---|---|
| exactly one unique sequence number per row | `ROW_NUMBER()` |
| ties share a rank and gaps are meaningful | `RANK()` |
| ties share a rank but ranks stay consecutive | `DENSE_RANK()` |
| divide ordered rows into roughly equal buckets | `NTILE(n)` |


## `NTILE()`

```sql
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(amount) AS total_paid
    FROM payment
    GROUP BY customer_id
)
SELECT
    customer_id,
    total_paid,
    NTILE(4) OVER (
        ORDER BY total_paid
    ) AS spending_quartile
FROM customer_totals
ORDER BY total_paid;
```

`NTILE(4)` divides ordered rows as evenly as possible into four buckets.


## Important NTILE interpretation

`NTILE(4)` creates four groups based on **row counts**, not four equal ranges of spending.

If spending values are highly skewed, the dollar ranges inside the four buckets can be very different.

This is segmentation by ordered position, not by equal-width numeric intervals.


# Part 6: The Top-N-per-Group Pattern


## A high-value analytical pattern

Business question:

> Who are the top three customers in each store by lifetime payments?

This requires two levels of logic:

1. compute customer totals,
2. rank those totals independently within each store.


## Step 1: aggregate to customer grain

```sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,
        c.store_id,
        SUM(p.amount) AS total_paid
    FROM customer c
    JOIN payment p
      ON c.customer_id = p.customer_id
    GROUP BY
        c.customer_id,
        c.first_name,
        c.last_name,
        c.store_id
)
SELECT *
FROM customer_totals;
```


## Step 2: rank inside each store

```sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,
        c.store_id,
        SUM(p.amount) AS total_paid
    FROM customer c
    JOIN payment p
      ON c.customer_id = p.customer_id
    GROUP BY
        c.customer_id,
        c.first_name,
        c.last_name,
        c.store_id
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY store_id
               ORDER BY total_paid DESC, customer_id
           ) AS store_rank
    FROM customer_totals
)
SELECT *
FROM ranked
ORDER BY store_id, store_rank;
```


## Step 3: filter in an outer query

```sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,
        c.store_id,
        SUM(p.amount) AS total_paid
    FROM customer c
    JOIN payment p
      ON c.customer_id = p.customer_id
    GROUP BY
        c.customer_id,
        c.first_name,
        c.last_name,
        c.store_id
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY store_id
               ORDER BY total_paid DESC, customer_id
           ) AS store_rank
    FROM customer_totals
)
SELECT *
FROM ranked
WHERE store_rank <= 3
ORDER BY store_id, store_rank;
```


## Why not filter the rank in the same WHERE clause?

This does not work:

```sql
SELECT
    ...,
    ROW_NUMBER() OVER (...) AS rn
FROM ...
WHERE rn <= 3;
```

The alias does not exist when `WHERE` is evaluated in that query block.

Use a CTE or derived table, then filter the already-computed ranking in the outer query.


## Quick check 5

**Think first.**

What are the three logical stages in a top-3-customers-per-store query?

<details>
<summary>Answer</summary>

Aggregate raw payments to customer totals, rank customer totals within each store, then filter the computed ranks to 3 or less in an outer query.

</details>


## `ROW_NUMBER` versus `RANK` for top N

If two customers tie for third place:

- `ROW_NUMBER()` still returns exactly three rows per store
- `RANK()` can return more than three rows because tied customers can share rank 3

Neither choice is automatically correct.

Choose based on whether the business requirement means:

> exactly N rows

or:

> everyone whose competitive rank is N or better


# Part 7: Window Frames


## Partition is not the same as frame

A **partition** is the broader group of rows available to the calculation.

A **frame** is the subset of that partition used for the current row's calculation.

Example:

```text
partition = all payments for customer 1
frame for row 5 = rows 3, 4, and 5
```

That distinction is essential for moving calculations.


## Frame syntax

A common form is:

```sql
ROWS BETWEEN frame_start AND frame_end
```

Useful boundaries include:

- `UNBOUNDED PRECEDING`
- `n PRECEDING`
- `CURRENT ROW`
- `n FOLLOWING`
- `UNBOUNDED FOLLOWING`


## Explicit running-total frame

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
        ROWS BETWEEN UNBOUNDED PRECEDING
                 AND CURRENT ROW
    ) AS running_total
FROM payment;
```

Each frame starts at the first row of the customer's partition and ends at the current row.


## Why write the frame explicitly?

PostgreSQL supplies a default frame when `ORDER BY` is present, but explicitly writing the frame is often clearer for teaching and review.

It also forces you to think about:

- ties in the order key,
- whether you want physical rows or value-based peer groups,
- whether the calculation is cumulative or moving.


## `ROWS` versus `RANGE`

For introductory analytical work, `ROWS` is often easier to reason about.

### `ROWS`
Counts physical rows relative to the current row.

```sql
ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
```

means:

> current row plus up to two previous rows.

### `RANGE`
Uses the ordering value and peer/value relationships rather than simply counting physical rows.

Because repeated order values can change `RANGE` behavior, use explicit `ROWS` when you want row-by-row movement.


# Part 8: Running Totals and Moving Averages


## Running total

A running total answers:

> How much has accumulated up to this row?

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
        ROWS BETWEEN UNBOUNDED PRECEDING
                 AND CURRENT ROW
    ) AS cumulative_spend
FROM payment;
```


## Real-world uses of running totals

- cumulative revenue by date
- account balance history
- inventory units consumed
- cumulative fundraising
- claims paid year-to-date
- project cost-to-date


## Moving average

A moving average uses only nearby rows.

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    AVG(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
        ROWS BETWEEN 2 PRECEDING
                 AND CURRENT ROW
    ) AS moving_avg_3
FROM payment
WHERE customer_id = 1
ORDER BY payment_date, payment_id;
```

The current frame contains at most three payment rows.


## What happens at the beginning?

For the first row:

```text
frame = first row only
```

For the second row:

```text
frame = first + second
```

For the third and later rows:

```text
frame = current + two preceding
```

PostgreSQL does not automatically return `NULL` until a full three-row frame exists.


## Requiring a complete moving window

If the analytical definition requires a full three observations, add a row count:

```sql
WITH x AS (
    SELECT
        customer_id,
        payment_id,
        payment_date,
        amount,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY payment_date, payment_id
        ) AS rn,
        AVG(amount) OVER (
            PARTITION BY customer_id
            ORDER BY payment_date, payment_id
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS moving_avg_3
    FROM payment
)
SELECT
    *,
    CASE
        WHEN rn >= 3 THEN moving_avg_3
        ELSE NULL
    END AS complete_moving_avg_3
FROM x;
```

This mirrors the textbook's discussion of suppressing incomplete early windows when the metric requires a full period.


## Quick check 6

**Think first.**

For `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`, how many rows can be in the frame once enough prior rows exist?

<details>
<summary>Answer</summary>

Three rows: the current row plus two preceding rows.

</details>


# Part 9: LAG() and LEAD()


## Adjacent-row access without a self-join

`LAG()` looks backward in the ordered partition.

`LEAD()` looks forward.

Basic patterns:

```sql
LAG(value)  OVER (PARTITION BY ... ORDER BY ...)
LEAD(value) OVER (PARTITION BY ... ORDER BY ...)
```


## Previous payment

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    LAG(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS previous_amount
FROM payment;
```

The first row of each customer partition has no previous row, so `previous_amount` is `NULL`.


## Change from previous payment

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    LAG(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS previous_amount,
    amount - LAG(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS change_from_previous
FROM payment;
```


## Offset and default value

```sql
LAG(amount, 2, 0)
```

means:

- look two rows back,
- return `0` if that row does not exist.

Be careful with defaults. `0` can imply "there was a prior value and it was zero," while `NULL` more honestly means "no prior row exists."


## LEAD for forward-looking comparisons

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    LEAD(payment_date) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS next_payment_date
FROM payment;
```

This can support questions such as:

> How long until this customer's next payment?


# Part 10: Period-over-Period Analysis


## Aggregate first, compare second

Monthly revenue comparison is a two-stage problem.

First produce one row per month:

```sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', payment_date) AS month,
        SUM(amount) AS revenue
    FROM payment
    GROUP BY DATE_TRUNC('month', payment_date)
)
SELECT *
FROM monthly_revenue
ORDER BY month;
```


## Add prior-month revenue with LAG

```sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', payment_date) AS month,
        SUM(amount) AS revenue
    FROM payment
    GROUP BY DATE_TRUNC('month', payment_date)
)
SELECT
    month,
    revenue,
    LAG(revenue) OVER (
        ORDER BY month
    ) AS previous_month_revenue
FROM monthly_revenue
ORDER BY month;
```


## Calculate the absolute change

```sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', payment_date) AS month,
        SUM(amount) AS revenue
    FROM payment
    GROUP BY DATE_TRUNC('month', payment_date)
),
with_previous AS (
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY month
        ) AS previous_revenue
    FROM monthly_revenue
)
SELECT
    month,
    revenue,
    previous_revenue,
    revenue - previous_revenue AS revenue_change
FROM with_previous
ORDER BY month;
```


## Calculate percentage change safely

```sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', payment_date) AS month,
        SUM(amount) AS revenue
    FROM payment
    GROUP BY DATE_TRUNC('month', payment_date)
),
with_previous AS (
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY month
        ) AS previous_revenue
    FROM monthly_revenue
)
SELECT
    month,
    revenue,
    previous_revenue,
    ROUND(
        100.0 * (revenue - previous_revenue)
        / NULLIF(previous_revenue, 0),
        2
    ) AS pct_change
FROM with_previous
ORDER BY month;
```

`NULLIF` from Module 6 prevents division by zero.


## Quick check 7

**Think first.**

Why do we usually aggregate to one row per month before using `LAG()` for month-over-month revenue?

<details>
<summary>Answer</summary>

Because `LAG()` moves between rows. To compare months, each row must first represent one month rather than one payment.

</details>


# Part 11: Named Windows


## Repeated window definitions become hard to read

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS running_sum,
    AVG(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS running_avg,
    COUNT(*) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS running_count
FROM payment;
```

The same definition is repeated three times.


## Define the window once

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    SUM(amount) OVER w AS running_sum,
    AVG(amount) OVER w AS running_avg,
    COUNT(*) OVER w AS running_count
FROM payment
WINDOW w AS (
    PARTITION BY customer_id
    ORDER BY payment_date, payment_id
);
```

A named window improves consistency and reduces repetitive code.


## Named windows reduce copy-paste errors

Suppose one repeated window accidentally uses:

```sql
ORDER BY payment_date
```

while another uses:

```sql
ORDER BY payment_date, payment_id
```

The measures can disagree because their row sequence is different.

A named window makes the shared analytical definition visible in one place.


# Part 12: Business Examples with Pagila


## Example 1: payment share of customer total

```sql
SELECT
    customer_id,
    payment_id,
    payment_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
    ) AS customer_total,
    ROUND(
        100.0 * amount
        / SUM(amount) OVER (
            PARTITION BY customer_id
        ),
        2
    ) AS pct_of_customer_total
FROM payment
ORDER BY customer_id, payment_date, payment_id;
```

This retains individual transactions while adding customer-level context.


## Example 2: rank films by rental count

```sql
WITH film_rentals AS (
    SELECT
        f.film_id,
        f.title,
        COUNT(*) AS rental_count
    FROM film f
    JOIN inventory i
      ON f.film_id = i.film_id
    JOIN rental r
      ON i.inventory_id = r.inventory_id
    GROUP BY f.film_id, f.title
)
SELECT
    title,
    rental_count,
    RANK() OVER (
        ORDER BY rental_count DESC
    ) AS rental_rank
FROM film_rentals
ORDER BY rental_rank, title;
```


## Example 3: top film per category

```sql
WITH category_film_rentals AS (
    SELECT
        c.name AS category,
        f.film_id,
        f.title,
        COUNT(*) AS rental_count
    FROM category c
    JOIN film_category fc
      ON c.category_id = fc.category_id
    JOIN film f
      ON fc.film_id = f.film_id
    JOIN inventory i
      ON f.film_id = i.film_id
    JOIN rental r
      ON i.inventory_id = r.inventory_id
    GROUP BY c.name, f.film_id, f.title
),
ranked AS (
    SELECT *,
           RANK() OVER (
               PARTITION BY category
               ORDER BY rental_count DESC
           ) AS category_rank
    FROM category_film_rentals
)
SELECT *
FROM ranked
WHERE category_rank = 1
ORDER BY category, title;
```

If films tie for first, `RANK()` keeps all tied winners.


## Example 4: cumulative monthly revenue

```sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', payment_date) AS month,
        SUM(amount) AS revenue
    FROM payment
    GROUP BY DATE_TRUNC('month', payment_date)
)
SELECT
    month,
    revenue,
    SUM(revenue) OVER (
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING
                 AND CURRENT ROW
    ) AS cumulative_revenue
FROM monthly_revenue
ORDER BY month;
```


## Example 5: three-period moving revenue average

```sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', payment_date) AS month,
        SUM(amount) AS revenue
    FROM payment
    GROUP BY DATE_TRUNC('month', payment_date)
)
SELECT
    month,
    revenue,
    AVG(revenue) OVER (
        ORDER BY month
        ROWS BETWEEN 2 PRECEDING
                 AND CURRENT ROW
    ) AS moving_avg_3
FROM monthly_revenue
ORDER BY month;
```


# Part 13: Real-World Window Patterns


## Customer analytics

Window functions support:

- customer rank within region
- cumulative lifetime value
- first/last transaction sequencing
- purchase-to-purchase change
- spending quartiles
- top-N customers per branch


## Finance

Typical window questions:

- running account balance
- month-over-month revenue
- rank securities within industry
- rolling volatility measures
- cumulative budget consumption
- prior-period balance comparison


## Healthcare

Typical window questions:

- visit sequence per patient
- time between visits
- rolling average wait time
- provider rank within clinic
- prior lab value
- change in measurement from previous encounter


## Operations

Typical window questions:

- cumulative units produced
- rank warehouses by throughput
- next shipment timestamp
- moving-average ticket volume
- top N cases per team
- time from one workflow event to the next


# Part 14: Common Errors and Misinterpretations


## Error 1: confusing GROUP BY with PARTITION BY

`GROUP BY` defines output rows.

`PARTITION BY` defines independent calculation windows without normally removing source rows.

If you use a window when you need one row per group, the result can contain many repeated summary values.

If you use `GROUP BY` when you need event detail, the individual rows disappear.


## Error 2: ranking the wrong grain

This ranks individual payment rows:

```sql
RANK() OVER (ORDER BY amount DESC)
```

It does not rank customers by lifetime spending.

Before ranking, make sure one row already represents the entity you intend to rank.


## Error 3: nondeterministic `ROW_NUMBER()`

Problem:

```sql
ROW_NUMBER() OVER (
    ORDER BY total_paid DESC
)
```

If totals tie, their relative row numbers are not fully defined.

More reproducible:

```sql
ROW_NUMBER() OVER (
    ORDER BY total_paid DESC, customer_id
)
```


## Error 4: filtering a window alias in the same WHERE

This is a common failure:

```sql
SELECT ...,
       ROW_NUMBER() OVER (...) AS rn
FROM ...
WHERE rn <= 3;
```

Move the ranking to a CTE or derived table, then filter the computed alias outside.


## Error 5: accidental cumulative calculation

These are not equivalent:

```sql
SUM(amount) OVER (
    PARTITION BY customer_id
)
```

and:

```sql
SUM(amount) OVER (
    PARTITION BY customer_id
    ORDER BY payment_date
)
```

The first gives the full customer total on every row.

The second becomes an ordered/running calculation.


## Error 6: final ORDER BY changes appearance, not the window logic

Changing:

```sql
ORDER BY customer_id, payment_date
```

at the end of the query does not rewrite:

```sql
OVER (ORDER BY payment_date)
```

The calculation was already defined by the window's internal ordering.


## Error 7: forgetting join fan-out

A window function does not correct duplicated rows created by a join.

If a payment appears twice after a join, then:

```sql
SUM(amount) OVER (...)
```

can count that duplicated amount twice.

Validate the input grain before the window runs.


## Error 8: assuming NTILE creates equal numeric ranges

`NTILE(4)` balances row counts as closely as possible.

It does not guarantee:

```text
0–25
25–50
50–75
75–100
```

or equal dollar-width spending bands.


## Error 9: ignoring incomplete early frames

A three-row moving average on the first row uses one row unless you explicitly suppress incomplete windows.

Whether that is acceptable depends on the business definition.


# Part 15: Debugging Window Queries


## A reliable window-function debugging sequence

1. write the base query without any window function
2. state the row grain
3. verify joins and row counts
4. add the simplest possible `OVER()`
5. add `PARTITION BY` and verify partition membership
6. add window `ORDER BY`
7. add an explicit frame if movement matters
8. add ranking or offset logic
9. only then add outer filters such as top-N


## Inspect one small partition

When a result is hard to understand, temporarily restrict the query:

```sql
WHERE customer_id IN (1, 2)
```

Then inspect:

- row order
- previous/next values
- cumulative totals
- frame boundaries

A small partition makes window behavior visible.


## Validate rankings with simple sorts

If you rank customer totals:

```sql
RANK() OVER (ORDER BY total_paid DESC)
```

also run:

```sql
SELECT *
FROM customer_totals
ORDER BY total_paid DESC;
```

The ranking should agree with the ordered totals.


# Part 16: Guided Practice


## Practice 1: full-table context

Return every payment with:

- payment ID
- amount
- grand total of all payment amounts

Use one window function.


<details>
<summary>Solution</summary>

```sql
SELECT
    payment_id,
    amount,
    SUM(amount) OVER () AS grand_total
FROM payment;
```

</details>


## Practice 2: customer context

Return every payment with:

- customer ID
- amount
- customer's total spending
- customer's average payment

Preserve one row per payment.


<details>
<summary>Solution</summary>

```sql
SELECT
    customer_id,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
    ) AS customer_total,
    AVG(amount) OVER (
        PARTITION BY customer_id
    ) AS customer_avg
FROM payment;
```

</details>


## Practice 3: payment sequence

For each customer, assign a deterministic sequential number to payments in chronological order.


<details>
<summary>Solution</summary>

```sql
SELECT
    customer_id,
    payment_id,
    payment_date,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS payment_number
FROM payment
ORDER BY customer_id, payment_number;
```

</details>


## Practice 4: customer ranking

Rank customers by total spending using `DENSE_RANK()`.

The result should have one row per customer.


<details>
<summary>Solution</summary>

```sql
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(amount) AS total_paid
    FROM payment
    GROUP BY customer_id
)
SELECT
    customer_id,
    total_paid,
    DENSE_RANK() OVER (
        ORDER BY total_paid DESC
    ) AS spending_rank
FROM customer_totals
ORDER BY spending_rank, customer_id;
```

</details>


## Practice 5: top three per store

Return exactly three customers from each store, ranked by total spending.

Use `ROW_NUMBER()`.


<details>
<summary>Solution</summary>

```sql
WITH customer_totals AS (
    SELECT
        c.customer_id,
        c.store_id,
        c.first_name,
        c.last_name,
        SUM(p.amount) AS total_paid
    FROM customer c
    JOIN payment p
      ON c.customer_id = p.customer_id
    GROUP BY
        c.customer_id,
        c.store_id,
        c.first_name,
        c.last_name
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY store_id
               ORDER BY total_paid DESC, customer_id
           ) AS store_rank
    FROM customer_totals
)
SELECT *
FROM ranked
WHERE store_rank <= 3
ORDER BY store_id, store_rank;
```

</details>


## Practice 6: moving average

For customer 1, calculate a moving average using the current payment plus the two previous payments.


<details>
<summary>Solution</summary>

```sql
SELECT
    customer_id,
    payment_id,
    payment_date,
    amount,
    AVG(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS moving_avg_3
FROM payment
WHERE customer_id = 1
ORDER BY payment_date, payment_id;
```

</details>


## Practice 7: prior payment

For each payment, return the prior payment amount for the same customer and calculate the difference.


<details>
<summary>Solution</summary>

```sql
SELECT
    customer_id,
    payment_id,
    payment_date,
    amount,
    LAG(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS prior_amount,
    amount - LAG(amount) OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
    ) AS amount_change
FROM payment;
```

</details>


## Practice 8: monthly running total

Aggregate to monthly revenue, then calculate cumulative revenue across months.


<details>
<summary>Solution</summary>

```sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', payment_date) AS month,
        SUM(amount) AS revenue
    FROM payment
    GROUP BY DATE_TRUNC('month', payment_date)
)
SELECT
    month,
    revenue,
    SUM(revenue) OVER (
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING
                 AND CURRENT ROW
    ) AS cumulative_revenue
FROM monthly_revenue
ORDER BY month;
```

</details>


# Part 17: AI Critique and Query Review


## AI-generated window SQL needs grain review

A model may produce valid-looking code with the wrong analytical unit.

Review window SQL by asking:

- What does one row represent before the window?
- Should the function operate on raw rows or pre-aggregated rows?
- Is the partition key correct?
- Is the window order deterministic?
- Are ties supposed to share a rank?
- Is the frame explicit enough?
- Does the outer filter preserve the intended top-N behavior?


## AI critique exercise 1

A model proposes:

```sql
SELECT
    c.store_id,
    c.customer_id,
    p.amount,
    RANK() OVER (
        PARTITION BY c.store_id
        ORDER BY p.amount DESC
    ) AS customer_rank
FROM customer c
JOIN payment p
  ON c.customer_id = p.customer_id;
```

It claims:

> "This ranks customers within each store by total spending."

What is wrong?


<details>
<summary>Critique</summary>

The query ranks individual payment rows by `p.amount`, not customers by lifetime spending.

A customer can appear many times.

The fix is to aggregate payments to one row per customer first, then rank the customer totals within `store_id`.

</details>


## AI critique exercise 2

A model proposes:

```sql
SELECT
    customer_id,
    payment_date,
    amount,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY payment_date
    ) AS rn
FROM payment;
```

What is the hidden reproducibility risk?


<details>
<summary>Critique</summary>

If a customer has more than one payment with the same `payment_date`, the ordering among those tied rows is not fully defined. Add a deterministic tiebreaker such as `payment_id`.

</details>


## AI critique exercise 3

A model says:

> "Adding an `ORDER BY` at the end of the query changes a running-total window."

Why is that incorrect?


<details>
<summary>Critique</summary>

The final `ORDER BY` changes the presentation order of the result. The running-total calculation is controlled by the `ORDER BY` inside the `OVER()` clause.

</details>


# Part 18: Knowledge Checks


## Knowledge check 1

**Think first.**

What keyword distinguishes a window function from a regular aggregate expression?

<details>
<summary>Answer</summary>

`OVER()`.

</details>


## Knowledge check 2

**Think first.**

Does `PARTITION BY` normally reduce the number of rows returned?

<details>
<summary>Answer</summary>

No. It defines independent calculation partitions but does not itself collapse rows.

</details>


## Knowledge check 3

**Think first.**

Which ranking function always assigns a unique sequence number?

<details>
<summary>Answer</summary>

`ROW_NUMBER()`.

</details>


## Knowledge check 4

**Think first.**

Which ranking function gives tied rows the same rank and leaves gaps afterward?

<details>
<summary>Answer</summary>

`RANK()`.

</details>


## Knowledge check 5

**Think first.**

Which ranking function gives tied rows the same rank without gaps?

<details>
<summary>Answer</summary>

`DENSE_RANK()`.

</details>


## Knowledge check 6

**Think first.**

What does `NTILE(4)` do?

<details>
<summary>Answer</summary>

It divides ordered rows into four buckets with row counts as balanced as possible and assigns bucket numbers 1 through 4.

</details>


## Knowledge check 7

**Think first.**

What is the purpose of `LAG()`?

<details>
<summary>Answer</summary>

It retrieves a value from a preceding row in the window's defined order.

</details>


## Knowledge check 8

**Think first.**

What does `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` define?

<details>
<summary>Answer</summary>

A frame containing the current row and up to two physical rows immediately before it.

</details>


## Knowledge check 9

**Think first.**

Why do top-N-per-group queries usually need an outer query or CTE?

<details>
<summary>Answer</summary>

Because the rank must be computed before it can be filtered by a WHERE condition.

</details>


## Knowledge check 10

**Think first.**

Can a bad join cause a correct window function to return misleading values?

<details>
<summary>Answer</summary>

Yes. The window operates on the rows produced by FROM/JOIN, so duplicated or otherwise incorrect input rows produce incorrect window results.

</details>


# Part 19: Concept Maps


## GROUP BY versus window functions

```text
GROUP BY
raw rows
   ↓
create groups
   ↓
aggregate
   ↓
one row per group


WINDOW FUNCTION
raw/result rows
   ↓
define partition
   ↓
define order
   ↓
define frame
   ↓
calculate context
   ↓
original rows remain + new analytical column
```


## Anatomy of a window expression

```text
SUM(amount)
    OVER (
        PARTITION BY customer_id
        ORDER BY payment_date, payment_id
        ROWS BETWEEN 2 PRECEDING
                 AND CURRENT ROW
    )
```

```text
SUM(amount)      = calculation
customer_id      = partition membership
payment_date...  = sequence
ROWS BETWEEN...  = current-row frame
```


## Choose the function by the question

```text
Need a group total beside every row? ---> SUM(...) OVER (PARTITION BY ...)

Need sequential IDs? ------------------> ROW_NUMBER()

Need competition ranking with gaps? ---> RANK()

Need tied ranking without gaps? --------> DENSE_RANK()

Need equal-count buckets? --------------> NTILE(n)

Need previous row's value? -------------> LAG()

Need next row's value? -----------------> LEAD()

Need running total? --------------------> SUM() OVER (ORDER BY ... frame)

Need moving average? -------------------> AVG() OVER (... ROWS BETWEEN ...)
```


# Part 20: Lab 6 Part II Readiness


## Before starting Lab 6 Part II

You should be able to:

- explain why window functions preserve rows
- write `OVER()` with and without `PARTITION BY`
- explain internal versus final `ORDER BY`
- use `ROW_NUMBER`, `RANK`, and `DENSE_RANK`
- explain tie behavior
- build a top-N-per-group query with CTEs
- calculate a running total
- calculate a moving average with `ROWS BETWEEN`
- use `LAG` for prior-row comparisons
- explain why pre-aggregation is necessary before ranking totals
- diagnose an incorrect window grain


## Lab validation habit

Add a comment above each major stage:

```sql
-- Grain after this CTE: one row per customer
```

Then:

```sql
-- Window partition: one independent ranking per store
```

Then:

```sql
-- Final result: top 3 customer rows per store
```

These comments make the logic reviewable before you submit.


## Lab-style challenge

Business question:

> Which three films have the highest rental counts within each film rating?

Build it in stages:

1. aggregate rentals to one row per film,
2. retain the film's rating,
3. rank films inside each rating,
4. return only ranks 1 through 3.


<details>
<summary>One solution</summary>

```sql
WITH film_rentals AS (
    SELECT
        f.film_id,
        f.title,
        f.rating,
        COUNT(*) AS rental_count
    FROM film f
    JOIN inventory i
      ON f.film_id = i.film_id
    JOIN rental r
      ON i.inventory_id = r.inventory_id
    GROUP BY f.film_id, f.title, f.rating
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY rating
               ORDER BY rental_count DESC, film_id
           ) AS rating_rank
    FROM film_rentals
)
SELECT *
FROM ranked
WHERE rating_rank <= 3
ORDER BY rating, rating_rank;
```

</details>


# Part 21: Interpreting Window Results


## Ranking is descriptive, not causal

A ranking tells you relative position under the measure you chose.

For example:

> Customer A ranks first in store 2 by lifetime spending.

It does not explain why Customer A spends more.

The ranking is a descriptive analytical result.


## Running totals depend on time order

A cumulative total has meaning only when the ordering variable represents the intended progression.

For time-based analysis, verify:

- timezone consistency
- duplicate timestamps
- missing dates
- whether the data contains every relevant period
- whether the ordering is event time or load time


## Moving averages depend on what a row represents

A three-row moving average is not always a three-day moving average.

If your rows are irregular transactions, three rows means three transactions.

For a true three-day or seven-day metric, you may need to aggregate to one row per day first or use a time-aware range definition.


## LAG compares adjacent rows, not necessarily adjacent calendar periods

If monthly data skips August, then September's `LAG()` value may be July.

The function compares the previous **row in the order**.

For reporting with missing periods, first construct or join to a complete calendar if the business definition requires every period.


# Part 22: Module Summary


## Key takeaways

- Window functions calculate across related rows while preserving the base result rows.
- `OVER()` distinguishes a window expression.
- `PARTITION BY` creates independent calculation groups.
- `ORDER BY` inside `OVER()` controls analytical sequence.
- Final-query `ORDER BY` controls display sequence.
- Aggregate functions such as `SUM`, `AVG`, and `COUNT` can operate as window functions.
- `ROW_NUMBER()` is unique; `RANK()` shares ties and leaves gaps; `DENSE_RANK()` shares ties without gaps.
- `NTILE(n)` divides ordered rows into roughly equal-count buckets.
- Top-N-per-group analysis usually requires pre-aggregation, ranking, then outer filtering.
- Window frames control which rows within the partition contribute to each row's calculation.
- `LAG()` and `LEAD()` make adjacent-row comparison possible without a self-join.
- Named windows reduce repetition and improve consistency.
- Correct window logic still depends on correct input grain and join cardinality.


## The question to carry forward

When you read or write a window function, ask:

> **What rows can this row see, in what order, and which of those rows are inside its current frame?**

If you can answer that clearly, most window expressions become understandable rather than mysterious.


## Up next: Module 9

Module 9 moves from analysis logic to data movement and practical database workflows.

The focus shifts toward:

- importing data
- exporting data
- PostgreSQL `COPY`
- bulk data workflows
- temporary and reusable structures

The query-building skills from Modules 1–8 remain important because imported data still has to be validated, transformed, and analyzed correctly.


## References

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for data analytics: Analyze data effectively, uncover insights and master advanced SQL for real-world applications* (4th ed.). Packt Publishing.

PostgreSQL Global Development Group. (n.d.). *Window functions*. PostgreSQL 16 Documentation.

PostgreSQL Global Development Group. (n.d.). *Window function calls*. PostgreSQL 16 Documentation.

Neon. (n.d.). *PostgreSQL window functions*.
